# Rede escolar de Minas Gerais — instância municipal — análise e preparação dos dados

INEP 2025 V2 · IBGE 2022 · SEE-MG para conferência (2026)

Três instâncias independentes: Fundamental — anos iniciais, Fundamental — anos
finais e Médio propedêutico. Execute as células em ordem. O notebook lê as fontes
originais e não depende do script Python.

Matrículas são observadas; demanda territorial e capacidades são estimativas.
As coordenadas escolares e a resolução do modelo ficam para a próxima etapa.
O estudo fica restrito a municípios de MG. Lavras é a instância inicial, não uma
alocação conjunta entre todos os municípios. A SEE não substitui matrículas do INEP.


## 1. Configuração

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
raiz = Path.cwd()
if raiz.name == "src":
    raiz = raiz.parent
codigo_municipio = "3138203"
# Deixe None para calibrar pelas matrículas da etapa. Totais externos exigem fonte.
totais_demanda = {"fundamental_iniciais": None, "fundamental_finais": None, "medio": None}
fatores_capacidade = [1.0, 1.1, 1.25, 1.5]
salvar_resultados = True

In [ ]:
etapas = {
    "fundamental_iniciais": ("Fundamental — anos iniciais", "QT_MAT_FUND_AI", "QT_TUR_FUND_AI"),
    "fundamental_finais": ("Fundamental — anos finais", "QT_MAT_FUND_AF", "QT_TUR_FUND_AF"),
    "medio": ("Médio — propedêutico", "QT_MAT_MED_PROP", "QT_TUR_MED_PROP"),
}

In [ ]:
if len(codigo_municipio) != 7 or not codigo_municipio.isdigit() or not codigo_municipio.startswith("31"):
    raise ValueError("Selecione um município de MG: código IBGE com sete dígitos e prefixo 31.")
saida = raiz / "outputs" / codigo_municipio / "escolas"
figuras = saida / "figuras"
if salvar_resultados:
    figuras.mkdir(parents=True, exist_ok=True)
print("Raiz:", raiz.resolve())
print("Saída:", saida)

## 2. Leitura das escolas

Leitura em blocos, com códigos em texto. Cada tabela deve ter uma linha por escola
no município. Fonte: [INEP — microdados](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar)
e dicionário XLSX do pacote 2025 V2.

No Médio usamos `QT_MAT_MED_PROP`: EJA, Educação Infantil, formação técnica e
Normal/Magistério ficam fora do recorte.

In [ ]:
def ler_municipio(arquivo, municipio, colunas):
    partes = []
    for bloco in pd.read_csv(arquivo, sep=";", encoding="latin1", dtype="string",
                             usecols=colunas, chunksize=20000):
        partes.append(bloco.loc[bloco.CO_MUNICIPIO.eq(municipio)])
    dados = pd.concat(partes, ignore_index=True)
    if dados.CO_ENTIDADE.isna().any() or dados.CO_ENTIDADE.duplicated().any():
        raise ValueError(f"Chave escolar ausente ou repetida: {arquivo}")
    return dados.drop(columns="CO_MUNICIPIO")

In [ ]:
arquivos = list((raiz / "data").rglob("Tabela_Escola_2025_V2.csv"))
if len(arquivos) != 1:
    raise ValueError("Esperado um único cadastro escolar 2025 V2 sob data/.")
pasta = arquivos[0].parent
colunas_cadastro = ["CO_MUNICIPIO", "CO_ENTIDADE", "NO_ENTIDADE", "NU_ANO_CENSO",
                    "TP_DEPENDENCIA", "TP_LOCALIZACAO", "TP_SITUACAO_FUNCIONAMENTO",
                    "QT_SALAS_UTILIZADAS", "IN_ESP_EXCLUSIVA_FUND_AI", "IN_ESP_EXCLUSIVA_FUND_AF"]

In [ ]:
cadastro = ler_municipio(arquivos[0], codigo_municipio, colunas_cadastro)
matriculas = ler_municipio(pasta / "Tabela_Matricula_2025_V2.csv", codigo_municipio,
                          ["CO_MUNICIPIO", "CO_ENTIDADE"] + [escolas[1] for escolas in etapas.values()])
turmas = ler_municipio(pasta / "Tabela_Turma_2025_V2.csv", codigo_municipio,
                      ["CO_MUNICIPIO", "CO_ENTIDADE"] + [escolas[2] for escolas in etapas.values()])

In [ ]:
if cadastro.empty:
    raise ValueError("Município sem escolas no cadastro.")
if not cadastro.NU_ANO_CENSO.eq("2025").all():
    raise ValueError("Ano inesperado no cadastro.")
base = cadastro.merge(matriculas, on="CO_ENTIDADE", how="left", validate="one_to_one")
base = base.merge(turmas, on="CO_ENTIDADE", how="left", validate="one_to_one")

In [ ]:
colunas_contagem = ["QT_SALAS_UTILIZADAS"] + [c for escolas in etapas.values() for c in escolas[1:]]
for coluna in colunas_contagem:
    base[coluna] = pd.to_numeric(base[coluna], errors="raise")
    valores = base[coluna].dropna()
    if valores.lt(0).any() or valores.mod(1).ne(0).any():
        raise ValueError(f"Contagem inválida: {coluna}")

In [ ]:
publica = base.TP_DEPENDENCIA.isin(["1", "2", "3"])
ativa = base.TP_SITUACAO_FUNCIONAMENTO.eq("1")
nomes_redes = {"1": "Federal", "2": "Estadual", "3": "Municipal", "4": "Privada"}
base["rede"] = base.TP_DEPENDENCIA.map(nomes_redes)
resumo_rede = base.loc[ativa].groupby("rede").agg(escolas_ativas=("CO_ENTIDADE", "size"))
display(resumo_rede)
print("Públicas ativas, todas as etapas:", int((publica & ativa).sum()))

### Conferência complementar com a SEE-MG

A fonte principal é o INEP 2025. A SEE-MG de 05/11/2026 informa modalidades
autorizadas e serve para localizar divergências, sem substituir registros de 2025.
O código estadual de seis dígitos recebe o prefixo 31 para cruzamento com o INEP.

In [ ]:
arquivo_see = raiz / "data/predios_escolas_sit_pivot.xlsx"
see = pd.read_excel(arquivo_see, header=7, dtype="string")
see = see.loc[see["Código da Escola"].notna()].copy()
see["CO_ENTIDADE"] = "31" + see["Código da Escola"].str.strip().str.zfill(6)
see_municipio = see["Código Município"].str.strip().str.zfill(4)
see = see.loc[see_municipio.eq(codigo_municipio[2:6])].copy()
if see.CO_ENTIDADE.duplicated().any():
    raise ValueError("Mais de um registro SEE por escola; revisar antes de cruzar.")
campos_see = {
    "fundamental_iniciais": "Ensino Regular - Ensino Fundamental - Anos iniciais",
    "fundamental_finais": "Ensino Regular - Ensino Fundamental - Anos finais",
    "medio": "Ensino Regular - Ensino Médio - Propedêutico",
}

In [ ]:
conferencia_see = base[["CO_ENTIDADE", "NO_ENTIDADE", "TP_SITUACAO_FUNCIONAMENTO"]].merge(
    see[["CO_ENTIDADE", *campos_see.values()]], on="CO_ENTIDADE", how="left",
    validate="one_to_one", indicator=True,
)
conferencia_see["presente_see_2026"] = conferencia_see._merge.eq("both")
for etapa, campo in campos_see.items():
    conferencia_see["autorizada_see_" + etapa] = conferencia_see[campo].eq("X").fillna(False)
display(conferencia_see[["CO_ENTIDADE", "NO_ENTIDADE", "presente_see_2026"]])

In [ ]:
exclusoes_documentadas = {
    "31347930": "CENAV: classes exclusivas no INEP 2025; apenas AEE na SEE 2026; fora da alocação geral",
}
# Paulo Lourenço permanece: oferta/matrículas observadas em 2025 não são
# substituídas pela ausência de autorização marcada na planilha de 2026.
base["exclusao_documentada"] = base.CO_ENTIDADE.map(exclusoes_documentadas)

## 3. Seleção e auditoria

Entram escolas públicas ativas com matrícula positiva na etapa. Contagens ausentes
ficam sinalizadas; não são convertidas em zero. Os campos `IN_COMUM_*` descrevem
atendimento de alunos da educação especial, por isso não são usados como filtro geral.
Unidades de atendimento especializado fora da alocação geral recebem exclusão
documentada. Uma divergência temporal na SEE não remove automaticamente a escola.


In [ ]:
auditorias = {}

for etapa, (_, campo_matriculas, campo_turmas) in etapas.items():
    auditoria = base.copy()
    positiva = base[campo_matriculas].gt(0).fillna(False)
    sem_contagem = base[[campo_matriculas, campo_turmas]].isna().any(axis=1)
    sem_turmas = base[campo_turmas].isna()

    if (publica & ativa & positiva & sem_turmas).any():
        raise ValueError(f"Matrículas positivas sem contagem de turmas: {etapa}")

    auditoria["selecionada"] = publica & ativa & positiva & base.exclusao_documentada.isna()
    auditoria["motivo_recorte"] = "sem matrículas na etapa"
    auditoria.loc[sem_contagem, "motivo_recorte"] = "contagem ausente; revisar"
    auditoria.loc[~ativa, "motivo_recorte"] = "fora de atividade"
    auditoria.loc[~publica, "motivo_recorte"] = "rede privada"
    auditoria.loc[auditoria.selecionada, "motivo_recorte"] = "selecionada"
    auditoria.loc[base.exclusao_documentada.notna(), "motivo_recorte"] = base.exclusao_documentada
    auditorias[etapa] = auditoria

In [ ]:
escolas_por_etapa = {}

for etapa, auditoria in auditorias.items():
    escolas = auditoria.loc[auditoria.selecionada].copy()
    escolas = escolas.sort_values("CO_ENTIDADE")
    if escolas.empty:
        raise ValueError(f"Nenhuma escola elegível em {etapa}.")

    _, campo_matriculas, campo_turmas = etapas[etapa]
    turmas_positivas = escolas[campo_turmas].where(escolas[campo_turmas].gt(0))
    escolas["matriculas_por_turma"] = escolas[campo_matriculas] / turmas_positivas
    escolas_por_etapa[etapa] = escolas

In [ ]:
linhas_resumo = []

for etapa, (nome, campo_matriculas, campo_turmas) in etapas.items():
    escolas = escolas_por_etapa[etapa]
    sem_contagem = base[[campo_matriculas, campo_turmas]].isna().any(axis=1)
    linhas_resumo.append({
        "etapa": etapa,
        "nome_etapa": nome,
        "escolas": len(escolas),
        "matriculas": int(escolas[campo_matriculas].sum()),
        "turmas": int(escolas[campo_turmas].sum()),
        "publicas_ativas_contagem_ausente": int((publica & ativa & sem_contagem).sum()),
    })

resumo_etapas = pd.DataFrame(linhas_resumo).set_index("etapa")
display(resumo_etapas)

In [ ]:
campos_etapas = [campo for _, matriculas, turmas in etapas.values()
                 for campo in (matriculas, turmas)]
sem_contagem = base[campos_etapas].isna().any(axis=1)
pendencias = base.loc[publica & ativa & sem_contagem]

display(pendencias[["CO_ENTIDADE", "NO_ENTIDADE", "rede"]])

In [ ]:
divergencias = []
for etapa, escolas in escolas_por_etapa.items():
    campo = "autorizada_see_" + etapa
    comparacao = escolas[["CO_ENTIDADE", "NO_ENTIDADE"]].merge(
        conferencia_see[["CO_ENTIDADE", "presente_see_2026", campo]], on="CO_ENTIDADE",
        validate="one_to_one",
    )
    comparacao["etapa"] = etapa
    divergencias.append(comparacao.loc[~comparacao.presente_see_2026 | ~comparacao[campo]])
divergencias_see = pd.concat(divergencias, ignore_index=True)
display(divergencias_see)

Uma escola pode aparecer em várias etapas. A união abaixo conta cada código INEP uma única vez.

In [ ]:
ofertas = pd.concat([escolas[["CO_ENTIDADE", "NO_ENTIDADE", "rede"]].assign(etapa=etapa)
                     for etapa, escolas in escolas_por_etapa.items()], ignore_index=True)
escolas_unicas = ofertas.drop_duplicates("CO_ENTIDADE").drop(columns="etapa")
print("Escolas distintas no recorte:", len(escolas_unicas))

In [ ]:
quadro_ofertas = pd.crosstab([ofertas.CO_ENTIDADE, ofertas.NO_ENTIDADE], ofertas.etapa).gt(0)
display(quadro_ofertas)
display(ofertas.groupby(["etapa", "rede"]).size().rename("escolas").unstack(fill_value=0))

## 4. Porte das escolas

Matrículas e turmas descrevem o porte observado. A razão matrículas/turma não é
capacidade máxima. Salas podem atender várias etapas e não são usadas como vagas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
resumo_etapas.set_index("nome_etapa").escolas.plot.bar(ax=axes[0], color="#2563eb", rot=15)
resumo_etapas.set_index("nome_etapa").matriculas.plot.bar(ax=axes[1], color="#059669", rot=15)
axes[0].set(title="Escolas elegíveis por etapa", xlabel="", ylabel="Escolas")
axes[1].set(title="Matrículas observadas por etapa — 2025", xlabel="", ylabel="Matrículas")
fig.tight_layout()
if salvar_resultados:
    fig.savefig(figuras / "comparacao_etapas.svg", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for ax, (etapa, (nome, campo_matriculas, campo_turmas)) in zip(axes, etapas.items()):
    escolas = escolas_por_etapa[etapa].nlargest(10, campo_matriculas).sort_values(campo_matriculas)
    ax.barh(escolas.NO_ENTIDADE, escolas[campo_matriculas], color="#2563eb")
    ax.set(title=nome, xlabel="Matrículas observadas (10 maiores)")
    ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
if salvar_resultados:
    fig.savefig(figuras / "porte_escolas.svg", bbox_inches="tight")
plt.show()

## 5. População dos setores censitários

Lemos apenas os atributos necessários para distribuir a demanda. Registros repetidos
são conferidos antes de manter a população uma vez por código. A representação
espacial e as coordenadas serão tratadas em uma etapa posterior.


In [ ]:
setores = gpd.read_file(raiz / "data/BR_setores_CD2022.gpkg",
                       layer="BR_setores_CD2022", where=f"CD_MUN = '{codigo_municipio}'",
                       columns=["CD_SETOR", "SITUACAO", "v0001"],
                       ignore_geometry=True)

In [ ]:
if setores.empty or setores.CD_SETOR.isna().any():
    raise ValueError("Recorte censitário vazio ou com identificadores ausentes.")
if setores.groupby("CD_SETOR")[["SITUACAO", "v0001"]].nunique(dropna=False).gt(1).any().any():
    raise ValueError("Atributos conflitantes em partes do mesmo setor.")


In [ ]:
setores = setores.drop_duplicates("CD_SETOR").copy()
if setores.v0001.isna().any() or setores.v0001.lt(0).any():
    raise ValueError("População ausente ou negativa.")
if setores.loc[~setores.SITUACAO.isin(["Urbana", "Rural"]), "v0001"].gt(0).any():
    raise ValueError("Há população fora do recorte urbano/rural.")
setores = setores.loc[setores.SITUACAO.isin(["Urbana", "Rural"]) & setores.v0001.gt(0)]
setores = setores.sort_values("CD_SETOR").reset_index(drop=True)

In [ ]:
demanda = setores[["CD_SETOR", "SITUACAO", "v0001"]].copy()
resumo_setores = demanda.groupby("SITUACAO").agg(
    setores=("CD_SETOR", "size"),
    populacao=("v0001", "sum"),
)
display(resumo_setores)


## 6. Idade da população por setor

Fonte: `Agregados_por_setores_demografia_BR.csv`, IBGE 2022. `V01032` conta moradores
de 5–9 anos, `V01033` de 10–14 e `V01034` de 15–19. Integramos pelo código do setor.
Valores `X` representam sigilo; permanecem ausentes na tabela de auditoria.

In [ ]:
arquivo_idades = raiz / "data/Agregados_por_setores_demografia_BR.csv"
campos_idade = {"V01032": "pop_5_9", "V01033": "pop_10_14", "V01034": "pop_15_19"}
colunas_idade = ["CD_setor", "V01006", *campos_idade]
blocos = pd.read_csv(arquivo_idades, sep=";", dtype="string",
                     usecols=colunas_idade, chunksize=20000)
recortes = []
for bloco in blocos:
    recortes.append(bloco.loc[bloco.CD_setor.str.startswith(codigo_municipio)])
idades = pd.concat(recortes, ignore_index=True)
idades = idades.rename(columns={"CD_setor": "CD_SETOR", **campos_idade})
if idades.CD_SETOR.isna().any() or idades.CD_SETOR.duplicated().any():
    raise ValueError("Códigos ausentes ou repetidos nos dados de idade.")

In [ ]:
colunas_numericas = ["V01006", *campos_idade.values()]
for coluna in colunas_numericas:
    valores = idades[coluna].str.strip()
    idades[coluna + "_suprimido"] = valores.str.upper().eq("X").fillna(False)
    valores = valores.mask(valores.str.upper().eq("X"))
    idades[coluna] = pd.to_numeric(valores, errors="raise")
    presentes = idades[coluna].dropna()
    if presentes.lt(0).any() or presentes.mod(1).ne(0).any():
        raise ValueError(f"Contagem etária inválida: {coluna}")

In [ ]:
demanda = demanda.merge(idades, on="CD_SETOR", how="left",
                        validate="one_to_one", indicator=True)
if not demanda._merge.eq("both").all():
    raise ValueError("Há setores de demanda sem correspondência no arquivo de idade.")
demanda = demanda.drop(columns="_merge")
comparaveis = demanda.V01006.notna()
diferencas = demanda.loc[comparaveis, "V01006"] - demanda.loc[comparaveis, "v0001"]
if diferencas.ne(0).any():
    raise ValueError("População das fontes diverge; revisar compatibilidade dos setores.")
ausencias_idade = demanda[list(campos_idade.values())].isna().sum().rename("setores_sem_valor")
display(ausencias_idade.to_frame())

### Valores suprimidos

Para completar as instâncias, estimamos **somente valores ausentes** pela população
total do setor multiplicada pela proporção daquela faixa nos setores com informação.
Preservamos a contagem original e marcamos cada preenchimento. É uma hipótese de
imputação, não recuperação do valor sigiloso. As somas etárias ficam sujeitas a essa
incerteza; nenhum `X` vira zero.

In [ ]:
taxas_idade = {}
for faixa in campos_idade.values():
    conhecidos = demanda[faixa].notna()
    taxa = demanda.loc[conhecidos, faixa].sum() / demanda.loc[conhecidos, "v0001"].sum()
    taxas_idade[faixa] = taxa
    demanda[faixa + "_imputado"] = ~conhecidos
    demanda[faixa + "_estimado"] = demanda[faixa].astype(float).fillna(demanda.v0001 * taxa)

display(pd.Series(taxas_idade, name="proporcao_nos_setores_conhecidos").to_frame())
colunas_imputadas = [faixa + "_imputado" for faixa in campos_idade.values()]
print("Setores com ao menos uma faixa estimada:", int(demanda[colunas_imputadas].any(axis=1).sum()))

### Grupos etários originais do IBGE

Mantemos os grupos de 5–9, 10–14 e 15–19 anos, sem repartir idades ou interpolar
faixas. Eles servem como referência demográfica para anos iniciais, anos finais e
Médio, respectivamente. Não coincidem exatamente com as idades esperadas das etapas.

A população é de 2022 e inclui potenciais alunos da rede privada. As contagens
originais e seus valores ausentes são preservados; somente as colunas de trabalho
usam preenchimentos identificados para valores sob sigilo.


In [ ]:
faixas_por_etapa = {
    "fundamental_iniciais": "pop_5_9",
    "fundamental_finais": "pop_10_14",
    "medio": "pop_15_19",
}
rotulos_faixas = {"pop_5_9": "5–9 anos", "pop_10_14": "10–14 anos", "pop_15_19": "15–19 anos"}
perfil_etario = pd.DataFrame({
    etapa: demanda[faixa + "_estimado"]
    for etapa, faixa in faixas_por_etapa.items()
})
if not np.isfinite(perfil_etario.to_numpy(dtype=float)).all():
    raise ValueError("Perfil etário incompleto.")
if perfil_etario.lt(0).any().any():
    raise ValueError("Perfil etário negativo.")


In [ ]:
comparacao_idades = resumo_etapas[["nome_etapa", "matriculas"]].copy()
comparacao_idades["faixa_ibge"] = pd.Series(faixas_por_etapa).map(rotulos_faixas)
comparacao_idades["populacao_observada_parcial_2022"] = pd.Series({
    etapa: demanda[faixa].sum() for etapa, faixa in faixas_por_etapa.items()
})
comparacao_idades["setores_com_valor_suprimido"] = pd.Series({
    etapa: int(demanda[faixa].isna().sum()) for etapa, faixa in faixas_por_etapa.items()
})
comparacao_idades["populacao_com_imputacao_2022"] = perfil_etario.sum()
comparacao_idades["razao_matriculas_2025_populacao_2022"] = (
    comparacao_idades.matriculas / comparacao_idades.populacao_com_imputacao_2022
)
display(comparacao_idades.round(3))
print("Somas observadas são parciais quando há sigilo. A razão não é taxa de atendimento.")


### Demanda potencial demográfica — cenário separado

Cada etapa usa diretamente seu grupo original do IBGE. Contagens observadas são
preservadas, valores suprimidos são imputados em colunas separadas. Para o modelo
inteiro, arredondamos cada valor de trabalho ao inteiro mais próximo; contagens
originais inteiras não mudam. Essa população inclui potenciais alunos da rede privada.
Não a interpretamos como demanda observada da rede pública.

In [ ]:
demandas_potenciais = {}
for etapa, faixa in faixas_por_etapa.items():
    potencial = demanda.copy()
    potencial["cenario_demanda"] = "potencial"
    potencial["faixa_ibge"] = rotulos_faixas[faixa]
    potencial["populacao_faixa_com_imputacao"] = perfil_etario[etapa]
    potencial["demanda_proporcional"] = perfil_etario[etapa]
    potencial["demanda_inteira"] = np.floor(perfil_etario[etapa] + 0.5).astype(int)
    demandas_potenciais[etapa] = potencial

## 7. Demanda calibrada para a rede pública

Distribuímos o total de matrículas públicas de cada etapa conforme seu perfil etário:
$w_{ie}=D_e A_{ie}/\sum_k A_{ke}$. O método dos maiores restos preserva o total inteiro.
O total coincide com as matrículas por calibração; a distribuição agora considera
idade. Mantemos a estimativa demográfica por setor para comparação.

Em `totais_demanda`, um total externo substitui o total de matrículas, preservando o
mesmo perfil espacial. A origem residencial observada dos estudantes é desconhecida.

In [ ]:
def maiores_restos(populacao, total):
    if total <= 0 or populacao.sum() <= 0:
        raise ValueError("Demanda e população totais precisam ser positivas.")
    proporcional = populacao.to_numpy(dtype=float) * total / populacao.sum()
    inteira = np.floor(proporcional).astype(int)
    ordem = np.argsort(-(proporcional - inteira), kind="stable")
    inteira[ordem[:total - int(inteira.sum())]] += 1
    return proporcional, inteira

In [ ]:
demandas = {}
for etapa, (_, campo_matriculas, _) in etapas.items():
    total = totais_demanda[etapa]
    if total is None:
        total = int(escolas_por_etapa[etapa][campo_matriculas].sum())
    if not isinstance(total, int) or isinstance(total, bool) or total <= 0:
        raise ValueError(f"Total de demanda inválido: {etapa}")
    demanda_etapa = demanda.copy()
    demanda_etapa["cenario_demanda"] = "calibrada"
    demanda_etapa["faixa_ibge"] = rotulos_faixas[faixas_por_etapa[etapa]]
    demanda_etapa["populacao_faixa_com_imputacao"] = perfil_etario[etapa]
    proporcional, inteira = maiores_restos(perfil_etario[etapa], total)
    demanda_etapa["demanda_proporcional"] = proporcional
    demanda_etapa["demanda_inteira"] = inteira
    assert int(inteira.sum()) == total
    demandas[etapa] = demanda_etapa

In [ ]:
for etapa, demanda_etapa in demandas.items():
    resumo_etapas.loc[etapa, "demanda_total"] = int(demanda_etapa.demanda_inteira.sum())
    resumo_etapas.loc[etapa, "origens_positivas"] = int(demanda_etapa.demanda_inteira.gt(0).sum())
    resumo_etapas.loc[etapa, "demanda_potencial"] = int(demandas_potenciais[etapa].demanda_inteira.sum())

display(resumo_etapas)


## 8. Cenários de capacidade e condições necessárias

Adotamos $Q_{je}=\lceil\alpha M_{je}\rceil$. Matrículas não equivalem a vagas oficiais;
os fatores são hipóteses. Para cada $p$, a soma das $p$ maiores capacidades precisa
cobrir a demanda, e cada setor deve caber em alguma escola. Esses critérios **não
comprovam viabilidade** de atribuição indivisível. Não somamos capacidades de diferentes
etapas como se fossem intercambiáveis.

In [ ]:
if any(not np.isfinite(fator) or fator <= 0 for fator in fatores_capacidade):
    raise ValueError("Fatores precisam ser finitos e positivos.")

capacidades_por_etapa = {}
for etapa, (_, campo_matriculas, _) in etapas.items():
    escolas = escolas_por_etapa[etapa]
    cenarios = []
    for fator in fatores_capacidade:
        capacidade = escolas[["CO_ENTIDADE"]].copy()
        capacidade["fator"] = fator
        capacidade["capacidade_hipotetica"] = np.ceil(
            escolas[campo_matriculas].astype(float) * fator
        ).astype(int)
        cenarios.append(capacidade)
    capacidades_por_etapa[etapa] = pd.concat(cenarios, ignore_index=True)

In [ ]:
diagnosticos_por_etapa = {}
for etapa, cenarios in capacidades_por_etapa.items():
    total = int(demandas[etapa].demanda_inteira.sum())
    maior_setor = int(demandas[etapa].demanda_inteira.max())
    diagnosticos = []
    for fator, cenario in cenarios.groupby("fator"):
        capacidades = cenario.capacidade_hipotetica.sort_values(ascending=False)
        somas = capacidades.cumsum().to_numpy()
        diagnostico = pd.DataFrame({
            "fator": fator,
            "p": np.arange(1, len(capacidades) + 1),
            "demanda_total": total,
            "soma_p_maiores_capacidades": somas,
            "passa_condicao_necessaria_total": somas >= total,
            "maior_setor_cabe": maior_setor <= capacidades.max(),
        })
        diagnosticos.append(diagnostico)
    diagnosticos_por_etapa[etapa] = pd.concat(diagnosticos, ignore_index=True)

A sensibilidade demográfica usa os mesmos portes escolares, mas demanda própria.
Os diagnósticos abaixo também são exportados separadamente.


In [ ]:
diagnosticos_potenciais = {}
for etapa, diagnostico in diagnosticos_por_etapa.items():
    potencial = demandas_potenciais[etapa]
    total = int(potencial.demanda_inteira.sum())
    tabela = diagnostico.copy()
    tabela["demanda_total"] = total
    tabela["passa_condicao_necessaria_total"] = tabela.soma_p_maiores_capacidades.ge(total)
    for fator, indices in tabela.groupby("fator").groups.items():
        capacidades = capacidades_por_etapa[etapa]
        maior = capacidades.loc[capacidades.fator.eq(fator), "capacidade_hipotetica"].max()
        tabela.loc[indices, "maior_setor_cabe"] = potencial.demanda_inteira.max() <= maior
    diagnosticos_potenciais[etapa] = tabela


In [ ]:
limites = []
for etapa, diagnostico in diagnosticos_por_etapa.items():
    for fator, cenario in diagnostico.groupby("fator"):
        quantidades_permitidas = cenario.loc[cenario.passa_condicao_necessaria_total & cenario.maior_setor_cabe, "p"]
        limites.append({"etapa": etapa, "fator": fator,
                        "menor_p_condicoes_necessarias": int(quantidades_permitidas.min()) if len(quantidades_permitidas) else pd.NA})
limites = pd.DataFrame(limites)
display(limites.pivot(index="etapa", columns="fator", values="menor_p_condicoes_necessarias"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (etapa, (nome, _, _)) in zip(axes, etapas.items()):
    diagnostico = diagnosticos_por_etapa[etapa]
    for fator, cenario in diagnostico.groupby("fator"):
        ax.plot(cenario.p, cenario.soma_p_maiores_capacidades / cenario.demanda_total, label=f"α={fator:g}")
    ax.axhline(1, color="black", linestyle="--", linewidth=1)
    ax.set(title=nome, xlabel="Escolas selecionadas (p)", ylabel="Capacidade das p maiores / demanda")
    ax.legend(fontsize=8)
fig.tight_layout()
if salvar_resultados:
    fig.savefig(figuras / "sensibilidade_capacidades.svg", bbox_inches="tight")
plt.show()

## 9. Exportação

As subpastas por etapa contêm as instâncias atuais. Metadados registram hipóteses e
fontes. Esta seção só grava arquivos se `salvar_resultados = True`. Os arquivos
históricos de instâncias na raiz de `escolas/` não são atualizados.

In [ ]:
metadados_comuns = {
    "uf": "MG", "codigo_uf": "31",
    "fonte_oferta": "INEP Censo Escolar 2025 V2",
    "conferencia_complementar": "SEE-MG, Sistema Prédios e Escolas, 05/11/2026",
    "municipio": codigo_municipio,
    "ano_inep": 2025,
    "ano_ibge": 2022,
    "escolas_cadastradas": len(base),
    "metodo_demanda": "perfil etário estimado por setor; calibração e maiores restos",
    "fonte_idades": arquivo_idades.name,
    "imputacao": "faixa ausente: população do setor * taxa nos setores conhecidos",
    "grupos_etarios": "originais do IBGE: 5–9, 10–14 e 15–19 anos; sem interpolação",
    "populacao": int(demanda.v0001.sum()),
    "distancias": "pendentes de coordenadas escolares verificadas",
    "capacidade": "hipótese ceil(fator * matrículas); não representa vagas oficiais",
    "viabilidade": "condições necessárias não comprovam viabilidade indivisível",
}

In [ ]:
resumos_exportacao = []
for etapa, (nome, campo_matriculas, campo_turmas) in etapas.items():
    escolas = escolas_por_etapa[etapa]
    demanda_etapa = demandas[etapa]
    resumo = metadados_comuns | {
        "etapa": etapa,
        "nome_etapa": nome,
        "campo_matriculas": campo_matriculas,
        "campo_turmas": campo_turmas,
        "escolas_selecionadas": len(escolas),
        "matriculas_observadas": int(escolas[campo_matriculas].sum()),
        "demanda_potencial_total": int(demandas_potenciais[etapa].demanda_inteira.sum()),
        "cenario_padrao": "calibrada",
        "demanda_total_cenario": int(demanda_etapa.demanda_inteira.sum()),
        "total_calibrado_pelas_matriculas": totais_demanda[etapa] is None,
        "faixa_ibge": rotulos_faixas[faixas_por_etapa[etapa]],
        "setores": len(demanda_etapa),
        "origens_positivas": int(demanda_etapa.demanda_inteira.gt(0).sum()),
    }
    resumos_exportacao.append(resumo)

In [ ]:
if salvar_resultados:
    for resumo in resumos_exportacao:
        etapa = resumo["etapa"]
        pasta_etapa = saida / etapa
        pasta_etapa.mkdir(parents=True, exist_ok=True)
        tabelas = {
            "escolas.csv": escolas_por_etapa[etapa],
            "auditoria_escolas.csv": auditorias[etapa],
            "demanda_setores.csv": demandas[etapa],  # Compatibilidade: cenário calibrado.
            "demanda_calibrada.csv": demandas[etapa],
            "demanda_potencial.csv": demandas_potenciais[etapa],
            "capacidades_cenarios.csv": capacidades_por_etapa[etapa],
            "diagnostico_capacidades.csv": diagnosticos_por_etapa[etapa],
            "diagnostico_calibrada.csv": diagnosticos_por_etapa[etapa],
            "diagnostico_potencial.csv": diagnosticos_potenciais[etapa],
        }
        for arquivo, tabela in tabelas.items():
            tabela.to_csv(pasta_etapa / arquivo, index=False)
        texto = json.dumps(resumo, ensure_ascii=False, indent=2)
        (pasta_etapa / "resumo.json").write_text(texto + "\n")

In [ ]:
if salvar_resultados:
    pd.DataFrame(resumos_exportacao).to_csv(saida / "resumo_etapas.csv", index=False)
    escolas_unicas.to_csv(saida / "escolas_unicas.csv", index=False)
    limites.to_csv(saida / "limites_necessarios_capacidade.csv", index=False)
    print("Tabelas e figuras exportadas para", saida)
    comparacao_idades.to_csv(saida / "comparacao_demografia_matriculas.csv")
    demanda.to_csv(saida / "auditoria_idades_setores.csv", index=False)

    conferencia_see.to_csv(saida / "conferencia_see_mg.csv", index=False)
    divergencias_see.to_csv(saida / "divergencias_see_mg.csv", index=False)


## 10. Síntese para a formulação

Em cada etapa, $I$ contém somente setores com demanda inteira positiva, $J$ contém
as escolas elegíveis, $w_i$ é a demanda inteira e $Q_j$ vem do cenário escolhido.
A função objetivo será $\min\sum_{i,j} w_i d_{ij}x_{ij}$. Impomos atribuição única,
vinculação $x_{ij}\leq y_j$, capacidade $\sum_iw_ix_{ij}\leq Q_jy_j$ e
$\sum_jy_j=p$. Todas as variáveis são binárias na versão indivisível.

Com toda a rede disponível, fixamos $y_j=1$. As três instâncias não compartilham
aberturas: um futuro modelo conjunto deverá representar etapas e capacidades
compatíveis. A quantidade de alunos estimada não identifica residências observadas.
Ainda faltam coordenadas verificadas, custos de distância e configuração do solver.

In [ ]:
sintese = resumo_etapas[["nome_etapa", "escolas", "matriculas", "demanda_total", "origens_positivas"]].copy()
sintese["pares_origem_escola"] = sintese.escolas * sintese.origens_positivas
sintese["variaveis_binarias_modelo_completo"] = sintese.pares_origem_escola + sintese.escolas
display(sintese)
print("Cada total de demanda foi preservado pelo arredondamento.")
print("Condições de capacidade não equivalem a solução viável ou ótima.")
print("Próxima etapa: completar e validar coordenadas escolares, calcular distâncias e resolver os modelos.")